# 05 — Era Maps
Focused dot-density maps split by era: **2018–2022** vs **2023–Present**.

Produces two map styles per era and data filter:
- **Clustered** — markers aggregate into numbered bubbles at low zoom (like the per-year maps)
- **Dots** — all points visible at every zoom, colored by incident type

Output files go to `output/maps/`. Run the **Archive** cell first to move old maps out of the way.

In [1]:
import shutil
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster

import config

In [2]:
# ── Era definitions ──────────────────────────────────────────────────────────
ERA_1_LABEL = "2018–2022"
ERA_1_YEARS = list(range(2018, 2023))
ERA_2_LABEL = "2023–Present"
ERA_2_YEARS = list(range(2023, 2027))

ERA_1_COLOR = "#2166ac"   # blue
ERA_2_COLOR = "#d6604d"   # red-orange

# Display name → exact NeighborhoodName value in the GeoJSON
NEIGHBORHOODS_OF_INTEREST = {
    "Parramore":      "Holden/Parramore",
    "Holden Heights": "Holden Heights",
    "Mercy Drive":    "Mercy Drive",
    "Carver Shores":  "Carver Shores",
}

NBD_HIGHLIGHT_COLOR = "#e31a1c"   # red outline for neighborhoods of interest


# ── Data loading ──────────────────────────────────────────────────────────────
def load_data() -> pd.DataFrame:
    df = pd.read_csv(config.DATA_PROCESSED / "orlando_incidents.csv", parse_dates=["date"])
    df = df.dropna(subset=["lat", "lon"])
    df["year"] = df["date"].dt.year
    print(f"Loaded {len(df):,} incidents | years: {sorted(df['year'].unique())}")
    return df

def load_neighborhoods() -> gpd.GeoDataFrame:
    return gpd.read_file(config.NEIGHBORHOODS_DIR / "orlando_neighborhoods.geojson").to_crs("EPSG:4326")


# ── Filters ───────────────────────────────────────────────────────────────────
def era1(df):        return df[df["year"].isin(ERA_1_YEARS)]
def era2(df):        return df[df["year"].isin(ERA_2_YEARS)]
def homicides(df):   return df[df["killed"] > 0]
def injury_only(df): return df[(df["injured"] > 0) & (df["killed"] == 0)]

In [3]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def incident_color(row) -> str:
    if row.get("mass") or row["killed"] >= 4:
        return "darkred"
    if row["killed"] > 0:
        return "red"
    if row["injured"] >= 3:
        return "orange"
    return "cadetblue"


def make_popup(row) -> str:
    date_str = row["date"].strftime("%b %d, %Y") if pd.notna(row["date"]) else "Unknown"
    return (
        f"<b>{date_str}</b><br>"
        f"{row.get('address', '')}<br>"
        f"Killed: {row['killed']} &nbsp;|&nbsp; Injured: {row['injured']}"
        + (" &nbsp;<b>⚠ Mass Shooting</b>" if row.get("mass") else "")
    )


def base_map(lat=config.ORLANDO_LAT, lon=config.ORLANDO_LON, zoom=config.DEFAULT_ZOOM) -> folium.Map:
    return folium.Map(location=[lat, lon], zoom_start=zoom, tiles="CartoDB positron")


def add_title(m, html_text):
    m.get_root().html.add_child(folium.Element(f"""
    <div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
         background:white;padding:8px 18px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:14px;text-align:center;">
        {html_text}
    </div>"""))


def add_legend(m):
    items = [
        ("Fatal (no mass)",  "red"),
        ("Multiple injured", "orange"),
        ("Injury only",      "cadetblue"),
        ("Mass shooting",    "darkred"),
    ]
    rows = "".join(
        f'<div><span style="color:{c};font-size:18px;line-height:1.4;">●</span>&nbsp;{label}</div>'
        for label, c in items
    )
    m.get_root().html.add_child(folium.Element(f"""
    <div style="position:fixed;bottom:30px;left:30px;background:white;
         padding:10px 14px;border-radius:6px;border:1px solid #aaa;
         z-index:1000;font-family:Arial,sans-serif;font-size:13px;">
        <b>Incident Type</b><br>{rows}
    </div>"""))


def save_map(m, filename):
    config.OUTPUT_MAPS.mkdir(parents=True, exist_ok=True)
    out = config.OUTPUT_MAPS / filename
    m.save(str(out))
    print(f"  Saved: {filename}")


def stats_line(df) -> str:
    return (f"{len(df):,} incidents &nbsp;|&nbsp; "
            f"{int(df['killed'].sum()):,} killed &nbsp;|&nbsp; "
            f"{int(df['injured'].sum()):,} injured")


def add_neighborhood_layers(m, neighborhoods_gdf: gpd.GeoDataFrame):
    """
    Adds two toggleable layers to m:
      - All Neighborhoods (thin gray outlines, off by default)
      - Neighborhoods of Interest (red outlines, on by default)
    """
    # All neighborhoods — off by default
    folium.GeoJson(
        neighborhoods_gdf.to_json(),
        name="All Neighborhoods",
        show=False,
        style_function=lambda x: {
            "color": "#888", "weight": 0.8, "fillOpacity": 0,
        },
        tooltip=folium.GeoJsonTooltip(fields=["NeighborhoodName"], aliases=["Neighborhood:"]),
    ).add_to(m)

    # Neighborhoods of interest — highlighted, on by default
    interest_names = list(NEIGHBORHOODS_OF_INTEREST.values())
    interest_gdf = neighborhoods_gdf[neighborhoods_gdf["NeighborhoodName"].isin(interest_names)]
    if not interest_gdf.empty:
        folium.GeoJson(
            interest_gdf.to_json(),
            name="Neighborhoods of Interest",
            show=True,
            style_function=lambda x: {
                "color": NBD_HIGHLIGHT_COLOR, "weight": 2.5, "fillOpacity": 0.06,
                "fillColor": NBD_HIGHLIGHT_COLOR,
            },
            tooltip=folium.GeoJsonTooltip(fields=["NeighborhoodName"], aliases=["Neighborhood:"]),
        ).add_to(m)

In [4]:
# ── Map type A: Clustered ─────────────────────────────────────────────────────

def make_clustered_map(df: pd.DataFrame, era_label: str, subtitle: str, filename: str,
                       split_layers: bool = False, neighborhoods_gdf=None):
    m = base_map()
    em_dash = "—"

    if neighborhoods_gdf is not None:
        add_neighborhood_layers(m, neighborhoods_gdf)

    if split_layers:
        fatal_cluster  = MarkerCluster(name="Fatal Shootings")
        injury_cluster = MarkerCluster(name="Injury Only")
        mass_layer     = folium.FeatureGroup(name="Mass Shootings (4+ victims)")
        for _, row in df.iterrows():
            popup = make_popup(row)
            dot = folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=6, fill=True, fill_opacity=0.75,
                color=incident_color(row),
                popup=folium.Popup(popup, max_width=300),
            )
            if row.get("mass") or row["killed"] >= 4:
                folium.Marker(
                    location=[row["lat"], row["lon"]],
                    icon=folium.Icon(color="black", icon="warning-sign", prefix="glyphicon"),
                    popup=folium.Popup(popup, max_width=300),
                ).add_to(mass_layer)
            elif row["killed"] > 0:
                dot.add_to(fatal_cluster)
            else:
                dot.add_to(injury_cluster)
        fatal_cluster.add_to(m)
        injury_cluster.add_to(m)
        mass_layer.add_to(m)
    else:
        cluster    = MarkerCluster(name="Incidents")
        mass_layer = folium.FeatureGroup(name="Mass Shootings (4+ victims)")
        for _, row in df.iterrows():
            popup = make_popup(row)
            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=6, fill=True, fill_opacity=0.75,
                color=incident_color(row),
                popup=folium.Popup(popup, max_width=300),
            ).add_to(cluster)
            if row.get("mass"):
                folium.Marker(
                    location=[row["lat"], row["lon"]],
                    icon=folium.Icon(color="black", icon="warning-sign", prefix="glyphicon"),
                    popup=folium.Popup(popup, max_width=300),
                ).add_to(mass_layer)
        cluster.add_to(m)
        mass_layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    subtitle_part = f" {em_dash} {subtitle}" if subtitle else ""
    add_title(m, f"<b>Orlando Gun Violence {era_label}{subtitle_part}</b><br>{stats_line(df)}")
    add_legend(m)
    save_map(m, filename)

In [5]:
# ── Map type B: All dots visible ──────────────────────────────────────────────

def make_dot_map(df: pd.DataFrame, era_label: str, subtitle: str, filename: str,
                 split_layers: bool = False, neighborhoods_gdf=None):
    m = base_map()
    em_dash = "—"

    if neighborhoods_gdf is not None:
        add_neighborhood_layers(m, neighborhoods_gdf)

    if split_layers:
        fatal_layer  = folium.FeatureGroup(name="Fatal Shootings")
        injury_layer = folium.FeatureGroup(name="Injury Only")
        mass_layer   = folium.FeatureGroup(name="Mass Shootings (4+ victims)")
        for _, row in df.iterrows():
            popup = make_popup(row)
            dot = folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=5, fill=True, fill_opacity=0.72,
                color=incident_color(row),
                popup=folium.Popup(popup, max_width=300),
            )
            if row.get("mass") or row["killed"] >= 4:
                folium.Marker(
                    location=[row["lat"], row["lon"]],
                    icon=folium.Icon(color="black", icon="warning-sign", prefix="glyphicon"),
                    popup=folium.Popup(popup, max_width=300),
                ).add_to(mass_layer)
            elif row["killed"] > 0:
                dot.add_to(fatal_layer)
            else:
                dot.add_to(injury_layer)
        fatal_layer.add_to(m)
        injury_layer.add_to(m)
        mass_layer.add_to(m)
    else:
        layer      = folium.FeatureGroup(name="All Incidents")
        mass_layer = folium.FeatureGroup(name="Mass Shootings (4+ victims)")
        for _, row in df.iterrows():
            popup = make_popup(row)
            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=5, fill=True, fill_opacity=0.72,
                color=incident_color(row),
                popup=folium.Popup(popup, max_width=300),
            ).add_to(layer)
            if row.get("mass"):
                folium.Marker(
                    location=[row["lat"], row["lon"]],
                    icon=folium.Icon(color="black", icon="warning-sign", prefix="glyphicon"),
                    popup=folium.Popup(popup, max_width=300),
                ).add_to(mass_layer)
        layer.add_to(m)
        mass_layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    subtitle_part = f" {em_dash} {subtitle}" if subtitle else ""
    add_title(m, f"<b>Orlando Gun Violence {era_label}{subtitle_part}</b><br>{stats_line(df)}")
    add_legend(m)
    save_map(m, filename)

# ── Archive old maps ─────────────────────────────────────────────────────────
# Moves per-year maps and older era maps to output/maps/archive/.
# Run this once to clean up, then comment it out.

def archive_old_maps():
    archive_dir = config.OUTPUT_MAPS / "archive"
    archive_dir.mkdir(parents=True, exist_ok=True)

    to_archive = [
        f for f in config.OUTPUT_MAPS.iterdir()
        if f.is_file() and (
            # per-year maps
            any(f.name.startswith(f"orlando_{y}") for y in range(2014, 2027))
            # old era dot/heat maps (superseded by this notebook)
            or f.name.startswith("dot_")
            or f.name.startswith("heat_")
            # old all-years maps
            or f.name.startswith("orlando_all_years")
            or f.name == "orlando_interactive_full.html"
        )
    ]

    for f in sorted(to_archive):
        shutil.move(str(f), str(archive_dir / f.name))
        print(f"  Archived: {f.name}")

    print(f"\nMoved {len(to_archive)} files to output/maps/archive/")


archive_old_maps()

In [6]:
# ── Neighborhood zoom maps ────────────────────────────────────────────────────
# Clips incidents to the neighborhood boundary (+ 400m buffer) and creates
# a zoomed-in map showing both eras as separate dot layers.

def make_neighborhood_zoom(df: pd.DataFrame, neighborhoods_gdf: gpd.GeoDataFrame,
                            display_name: str, dataset_name: str):
    nbd = neighborhoods_gdf[neighborhoods_gdf["NeighborhoodName"] == dataset_name]
    if nbd.empty:
        print(f"  '{dataset_name}' not found in neighborhood file — skipping.")
        return

    bounds = nbd.geometry.total_bounds   # [minx, miny, maxx, maxy]
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

    # Clip incidents to neighborhood + 400m buffer
    nbd_buf = nbd.to_crs("EPSG:32617").copy()
    nbd_buf["geometry"] = nbd_buf.geometry.buffer(400)
    nbd_buf = nbd_buf.to_crs("EPSG:4326")
    inc_gdf = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326"
    )
    clipped = gpd.sjoin(inc_gdf, nbd_buf[["geometry"]], how="inner", predicate="within")
    local_df = pd.DataFrame(clipped.drop(columns=["geometry", "index_right"], errors="ignore"))

    m = base_map(lat=center_lat, lon=center_lon, zoom=14)

    # Neighborhood boundary
    folium.GeoJson(
        nbd.to_json(),
        name=f"{display_name} boundary",
        style_function=lambda x: {
            "color": NBD_HIGHLIGHT_COLOR, "weight": 2.5, "fillOpacity": 0.05,
            "fillColor": NBD_HIGHLIGHT_COLOR,
        },
    ).add_to(m)

    # Era 1 incidents
    e1_local = era1(local_df)
    e1_layer = folium.FeatureGroup(name=f"{ERA_1_LABEL} ({len(e1_local):,})")
    for _, row in e1_local.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6, fill=True, fill_opacity=0.8,
            color=incident_color(row),
            popup=folium.Popup(make_popup(row), max_width=280),
        ).add_to(e1_layer)
    e1_layer.add_to(m)

    # Era 2 incidents
    e2_local = era2(local_df)
    e2_layer = folium.FeatureGroup(name=f"{ERA_2_LABEL} ({len(e2_local):,})")
    for _, row in e2_local.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6, fill=True, fill_opacity=0.8,
            color=incident_color(row),
            popup=folium.Popup(make_popup(row), max_width=280),
        ).add_to(e2_layer)
    e2_layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    add_title(m,
        f"<b>{display_name}</b><br>"
        f"<small style='font-weight:normal;'>"
        f"{ERA_1_LABEL}: {len(e1_local):,} incidents &nbsp;|&nbsp; "
        f"{ERA_2_LABEL}: {len(e2_local):,} incidents</small>"
    )
    add_legend(m)

    safe = display_name.lower().replace(" ", "_").replace("/", "_")
    save_map(m, f"zoom_{safe}.html")

In [7]:
# ── Side-by-side synchronized map ────────────────────────────────────────────
# Left map = Era 1 (2018–2022), Right map = Era 2 (2023–Present).
# Pan and zoom are linked — both sides move together.
# clustered=False → raw dots colored by incident type
# clustered=True  → markers aggregate into numbered bubbles at low zoom

from folium.plugins import DualMap

def make_side_by_side_map(df: pd.DataFrame, subtitle: str, filename: str,
                           neighborhoods_gdf=None, clustered: bool = False):
    m = DualMap(
        location=[config.ORLANDO_LAT, config.ORLANDO_LON],
        zoom_start=config.DEFAULT_ZOOM,
        tiles="CartoDB positron",
    )

    e1_df = era1(df)
    e2_df = era2(df)
    em_dash = "—"

    # Neighborhood overlays on both sides
    if neighborhoods_gdf is not None:
        add_neighborhood_layers(m.m1, neighborhoods_gdf)
        add_neighborhood_layers(m.m2, neighborhoods_gdf)

    def add_incidents(sub_df, target_map):
        if clustered:
            container = MarkerCluster(name="Incidents")
        else:
            container = folium.FeatureGroup(name="Incidents")
        for _, row in sub_df.iterrows():
            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=5 if not clustered else 6,
                fill=True,
                fill_opacity=0.75,
                color=incident_color(row),
                popup=folium.Popup(make_popup(row), max_width=280),
            ).add_to(container)
        container.add_to(target_map)

    add_incidents(e1_df, m.m1)
    add_incidents(e2_df, m.m2)

    def era_stat_box(label, sub_df, side):
        anchor = "left:10px" if side == "left" else "right:10px"
        return f"""
        <div style="position:fixed;top:10px;{anchor};
             background:white;padding:6px 14px;border-radius:6px;border:1px solid #aaa;
             z-index:9999;font-family:Arial,sans-serif;font-size:13px;">
            <b>{label}</b><br>
            <span style="font-size:11px;">
                {len(sub_df):,} incidents &nbsp;|&nbsp;
                {int(sub_df['killed'].sum()):,} killed &nbsp;|&nbsp;
                {int(sub_df['injured'].sum()):,} injured
            </span>
        </div>"""

    m.m1.get_root().html.add_child(folium.Element(era_stat_box(ERA_1_LABEL, e1_df, "left")))
    m.m2.get_root().html.add_child(folium.Element(era_stat_box(ERA_2_LABEL, e2_df, "right")))

    # Centered title across both maps
    subtitle_part = f" {em_dash} {subtitle}" if subtitle else ""
    m.m1.get_root().html.add_child(folium.Element(f"""
    <div style="position:fixed;top:10px;left:50vw;transform:translateX(-50%);
         background:white;padding:8px 18px;border-radius:6px;border:1px solid #aaa;
         z-index:9999;font-family:Arial,sans-serif;font-size:14px;
         text-align:center;pointer-events:none;white-space:nowrap;">
        <b>Orlando Gun Violence{subtitle_part} — Era Comparison</b>
    </div>"""))

    # Vertical divider between the two maps
    m.m1.get_root().html.add_child(folium.Element("""
    <div style="position:fixed;top:0;left:calc(50% - 2px);width:4px;height:100vh;
         background:#222;z-index:9998;pointer-events:none;"></div>"""))

    add_legend(m.m1)
    save_map(m, filename)

In [8]:
# ── Generate all maps ─────────────────────────────────────────────────────────

df  = load_data()
nbd = load_neighborhoods()
e1  = era1(df)
e2  = era2(df)

print(f"\n  Era 1 ({ERA_1_LABEL}): {len(e1):,} incidents")
print(f"  Era 2 ({ERA_2_LABEL}): {len(e2):,} incidents")

# ── All incidents (split Fatal / Injury / Mass, with neighborhood overlay) ────
print("\n── All Incidents ──────────────────────────────────────")
make_clustered_map(e1, ERA_1_LABEL, "", "era1_all_clustered.html", split_layers=True,  neighborhoods_gdf=nbd)
make_dot_map(      e1, ERA_1_LABEL, "", "era1_all_dots.html",      split_layers=True,  neighborhoods_gdf=nbd)
make_clustered_map(e2, ERA_2_LABEL, "", "era2_all_clustered.html", split_layers=True,  neighborhoods_gdf=nbd)
make_dot_map(      e2, ERA_2_LABEL, "", "era2_all_dots.html",      split_layers=True,  neighborhoods_gdf=nbd)

# ── Fatal shootings ───────────────────────────────────────────────────────────
print("\n── Fatal Shootings ─────────────────────────────────────")
make_clustered_map(homicides(e1), ERA_1_LABEL, "Fatal", "era1_homicides_clustered.html", neighborhoods_gdf=nbd)
make_dot_map(      homicides(e1), ERA_1_LABEL, "Fatal", "era1_homicides_dots.html",      neighborhoods_gdf=nbd)
make_clustered_map(homicides(e2), ERA_2_LABEL, "Fatal", "era2_homicides_clustered.html", neighborhoods_gdf=nbd)
make_dot_map(      homicides(e2), ERA_2_LABEL, "Fatal", "era2_homicides_dots.html",      neighborhoods_gdf=nbd)

# ── Injury shootings ──────────────────────────────────────────────────────────
print("\n── Injury Shootings (Non-Fatal) ────────────────────────")
make_clustered_map(injury_only(e1), ERA_1_LABEL, "Injury", "era1_injury_clustered.html", neighborhoods_gdf=nbd)
make_dot_map(      injury_only(e1), ERA_1_LABEL, "Injury", "era1_injury_dots.html",      neighborhoods_gdf=nbd)
make_clustered_map(injury_only(e2), ERA_2_LABEL, "Injury", "era2_injury_clustered.html", neighborhoods_gdf=nbd)
make_dot_map(      injury_only(e2), ERA_2_LABEL, "Injury", "era2_injury_dots.html",      neighborhoods_gdf=nbd)

# ── Side-by-side synchronized comparison maps ─────────────────────────────────
print("\n── Side-by-Side Era Comparisons ────────────────────────")
make_side_by_side_map(df, "",       "sidebyside_all.html",             neighborhoods_gdf=nbd)
make_side_by_side_map(df, "",       "sidebyside_all_clustered.html",   neighborhoods_gdf=nbd, clustered=True)
make_side_by_side_map(homicides(df),  "Fatal",  "sidebyside_homicides.html", neighborhoods_gdf=nbd)
make_side_by_side_map(injury_only(df),"Injury", "sidebyside_injury.html",    neighborhoods_gdf=nbd)

# ── Neighborhood zoom maps ────────────────────────────────────────────────────
print("\n── Neighborhood Zooms ──────────────────────────────────")
for display_name, dataset_name in NEIGHBORHOODS_OF_INTEREST.items():
    make_neighborhood_zoom(df, nbd, display_name, dataset_name)

print("\nDone. Maps saved to output/maps/")

Loaded 1,742 incidents | years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

  Era 1 (2018–2022): 750 incidents
  Era 2 (2023–Present): 331 incidents

── All Incidents ──────────────────────────────────────
  Saved: era1_all_clustered.html
  Saved: era1_all_dots.html
  Saved: era2_all_clustered.html
  Saved: era2_all_dots.html

── Fatal Shootings ─────────────────────────────────────
  Saved: era1_homicides_clustered.html
  Saved: era1_homicides_dots.html
  Saved: era2_homicides_clustered.html
  Saved: era2_homicides_dots.html

── Injury Shootings (Non-Fatal) ────────────────────────
  Saved: era1_injury_clustered.html
  Saved: era1_injury_dots.html
  Saved: era2_injury_clustered.html
  Saved: era2_injury_dots.html

── Side-by-Side Era Comparisons ────────────────────────
  Saved: sidebyside_all.html
  Saved: sidebyside_all